# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR⁲ dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields.

In [ ]:
# List all available record sets, their @id and field @id's
print("Record sets and their fields:")
record_set_ids = []
for record_set in metadata.record_sets:
    print(f"\nRecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If there are record sets, load them into DataFrames
if not record_set_ids:
    raise ValueError("No record sets found in the dataset.")

dataframes = {}
for record_set_id in record_set_ids:
    # Load records for each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# Pick the first record set for further exploration
selected_record_set_id = record_set_ids[0]
if selected_record_set_id in dataframes:
    print(f"\nFields in DataFrame (columns):\n{dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())
else:
    raise ValueError(f"No data available for record set {selected_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping by key attributes.

In [ ]:
# EDA on the selected record set
df = dataframes[selected_record_set_id]

# Pick a numeric field (guessing column names typical for mass-spec data)
possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
if not possible_numeric_fields:
    # Maybe the DataFrame has all strings (CSV load issue): Try to infer
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f\
"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optional: group by a likely categorical field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            if df[col].nunique() < df.shape[0] // 2:
                group_field = col
                break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields found in the record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric_fields:
    # Histogram of the selected numeric_field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If group_field is available, barplot group means
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(12,5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields found to visualize.")

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR⁲ dataset using Croissant metadata and the `mlcroissant` Python library. Further analysis could include domain-specific protein statistics or linking to UniProt identifiers, depending on your research questions.